In [1]:
# Source libraries
import os
import sys
import logging
import subprocess
from datetime import datetime, timedelta
import pandas as pd
# pip install apache-airflow
from airflow import DAG
from airflow.utils.dates import days_ago
from airflow.operators.bash import BashOperator
from airflow.operators.python import PythonOperator
# pip install google-cloud-storage
from google.cloud import storage
# pip install apache-airflow-providers-google
from airflow.providers.google.cloud.operators.bigquery import BigQueryCreateExternalTableOperator
#import pyarrow as pa
import pyarrow.csv as pv
import pyarrow.parquet as pq
# pip install pyspark
import pyspark
from pyspark.sql import SparkSession, Row
from pyspark.conf import SparkConf
from pyspark.context import SparkContext
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import pyspark.sql.types as T


In [2]:
# Get GCP input data
PROJECT_ID = 'intricate-reef-411403'
BUCKET = 'project_bucket-intricate-reef-411403'
credentials_location = '/home/jdelzio/de-bootcamp-project/.google/credentials/gcp.json'
DATASET = 'project'
PROJECT_PATH = '/home/jdelzio/de-bootcamp-project'

# Get file structure data
local_data_path = PROJECT_PATH+"/data/raw/Mendeley_data/"
temp_path = PROJECT_PATH+"/data/raw/temp/"
local_data_file = "100_Batches_IndPenSim_V3.csv"
gcs_input_path = "raw/"
gcs_output_path = "processed/sample_context/"
gcs_raman_path = "processed/raman_context/"
spark_jar_path = PROJECT_PATH+"/lib/gcs-connector-hadoop3-2.2.5.jar,"+PROJECT_PATH+"/lib/spark-3.5-bigquery-0.37.0.jar"
execution_time = datetime.utcnow()

In [3]:
start_spark_master = "cd $SPARK_HOME && ./sbin/start-master.sh --port 7078"
start_spark_worker = "cd $SPARK_HOME && ./sbin/start-worker.sh spark://127.0.0.1:7078"
start_master_process = subprocess.Popen(start_spark_master, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
start_master_output, start_master_error = start_master_process.communicate()
start_worker_process = subprocess.Popen(start_spark_worker, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
start_worker_output, start_worker_error = start_worker_process.communicate()

In [4]:
conf = SparkConf() \
        .setMaster("spark://127.0.0.1:7078") \
        .setAppName("process_raw_data") \
        .set("spark.jars", spark_jar_path) \
        .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
        .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [5]:
# set up spark context
sc = SparkContext(conf=conf)
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

24/04/18 02:27:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [6]:
# Start Spark session using standalone cluster
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [10]:
# Gather Data
df= spark.read.parquet(f'gs://{BUCKET}/{gcs_input_path}*.parquet') \
    .select(["id","Time (h)"])
# sort by id, save row count
df = df.orderBy("id")
nrows = df.count()

In [11]:
# Find new batch start indeces
batch_start_df = df \
    .filter(df["Time (h)"] == 0.2) \
    .select("id") \
    .withColumnRenamed("id","batch_start_id") \
    .withColumn("Batch Number",F.monotonically_increasing_id()+1)

In [12]:
# Add next batch id for join clause
window_frame = Window.orderBy("batch_start_id")
batch_start_df = batch_start_df.withColumn("next_batch_start_id", F.lead("batch_start_id").over(window_frame))
# fill final next_batch_start_id with nrow df_sorted + 1
batch_start_df = batch_start_df.fillna(nrows+1)
# join batch number context to df
df_processed = df.join(batch_start_df, (df.id >= batch_start_df.batch_start_id) & (df.id < batch_start_df.next_batch_start_id ), "inner") \
    .drop(*["batch_start_id","next_batch_start_id"])

In [13]:
completed_batches = 30
first_new_batch = df_processed \
    .filter(df_processed["Batch Number"] == completed_batches+1) \
    .select("id") \
    .head()[0]

24/04/18 02:29:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:29:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:29:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:29:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:29:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:29:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 0

In [14]:
ts_current = datetime.utcnow()
ts_first_30_batches = [ts_current - i*timedelta(seconds=0.06) for i in range(1,first_new_batch)]
ts_first_30_batches.reverse()
ts_last_70_batches = [ts_current + i*timedelta(seconds=0.06) for i in range(1,nrows-first_new_batch+2)]
sample_ts = ts_first_30_batches
sample_ts.extend(ts_last_70_batches)
# join sample ts to processed_df
sample_ts_df = spark.createDataFrame([Row(index=i+1, sample_ts=sample_ts[i]) for i in range(nrows)])
df_batch_context = df_processed.join(sample_ts_df, df_processed.id == sample_ts_df.index, "inner") \
    .drop("index")

In [17]:
df_batch_context \
    .repartition(4) \
    .write.mode('overwrite').parquet(f'gs://{BUCKET}/{gcs_output_path}')

24/04/18 02:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:31:02 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:31:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:31:10 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:31:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 02:31:12 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
24/04/18 0

In [18]:
spark.stop()

In [19]:
# Stop Local Standalone cluster
!cd $SPARK_HOME && ./sbin/stop-master.sh

IOStream.flush timed out
stopping org.apache.spark.deploy.master.Master


In [20]:
# Stop Worker
!cd $SPARK_HOME && ./sbin/stop-worker.sh

IOStream.flush timed out
stopping org.apache.spark.deploy.worker.Worker


[2024-04-18T02:32:28.150+0000] {clientserver.py:505} INFO - Error while sending or receiving.
Traceback (most recent call last):
  File "/home/jdelzio/de-bootcamp-project/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer
[2024-04-18T02:32:28.156+0000] {clientserver.py:543} INFO - Closing down clientserver connection
[2024-04-18T02:32:28.158+0000] {java_gateway.py:1052} INFO - Exception while sending command.
Traceback (most recent call last):
  File "/home/jdelzio/de-bootcamp-project/.venv/lib/python3.11/site-packages/py4j/clientserver.py", line 503, in send_command
    self.socket.sendall(command.encode("utf-8"))
ConnectionResetError: [Errno 104] Connection reset by peer

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/jdelzio/de-bootcamp-project/.venv/lib/python3.11/site-pack